In [2]:
import fastf1
import pandas as pd
import numpy as np
import requests
import os
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

In [3]:

load_dotenv()

True

In [4]:

fastf1.Cache.enable_cache("f1_cache")

In [5]:
# Load 2024 Japanese GP race session, lap and sector times
session_2024 = fastf1.get_session(2024, "Japan", "R")
session_2024.load()
laps_2024 = session_2024.laps[["Driver", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].copy()
laps_2024.dropna(inplace=True)

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.6.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No c

In [6]:
for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps_2024[f"{col} (s)"] = laps_2024[col].dt.total_seconds()

In [7]:
sector_times_2024 = laps_2024.groupby("Driver")[["Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)"]].mean().reset_index()

In [8]:
qualifying_2025 = pd.DataFrame({
    "Driver": ["VER", "NOR", "PIA", "LEC", "RUS", "HAM", "GAS", "ALO", "TSU", "SAI", "HUL", "OCO", "STR"],
    "QualifyingTime (s)": [86.983, 86.995, 87.027, 87.299, 87.318, 87.610, 87.822, 87.897, 88.000, 87.836, 88.570, 88.696, 89.271]
})

In [9]:
driver_wet_performance = {
    "VER": 0.975196, 
    "HAM": 0.976464,  
    "LEC": 0.975862,  
    "NOR": 0.978179,  
    "ALO": 0.972655,  
    "RUS": 0.968678,  
    "SAI": 0.978754,  
    "TSU": 0.996338,  
    "OCO": 0.981810,  
    "GAS": 0.978832,  
    "STR": 0.979857   
}
qualifying_2025["WetPerformanceFactor"] = qualifying_2025["Driver"].map(driver_wet_performance)

In [10]:
API_KEY = os.getenv('OPENWEATHER_API_KEY')

if not API_KEY:
    raise ValueError("OPENWEATHER_API_KEY not found in environment variables. Please check your .env file.")

In [11]:

weather_url = f"http://api.openweathermap.org/data/2.5/forecast?lat=34.8823&lon=136.5845&appid={API_KEY}&units=metric"
response = requests.get(weather_url)
weather_data = response.json()
print(weather_data)

{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1754708400, 'main': {'temp': 27.94, 'feels_like': 29.81, 'temp_min': 27.94, 'temp_max': 28.41, 'pressure': 1006, 'sea_level': 1006, 'grnd_level': 998, 'humidity': 64, 'temp_kf': -0.47}, 'weather': [{'id': 500, 'main': 'Rain', 'description': 'light rain', 'icon': '10d'}], 'clouds': {'all': 95}, 'wind': {'speed': 1.07, 'deg': 143, 'gust': 0.96}, 'visibility': 10000, 'pop': 0.9, 'rain': {'3h': 1.17}, 'sys': {'pod': 'd'}, 'dt_txt': '2025-08-09 03:00:00'}, {'dt': 1754719200, 'main': {'temp': 28.44, 'feels_like': 30.8, 'temp_min': 28.44, 'temp_max': 29.44, 'pressure': 1006, 'sea_level': 1006, 'grnd_level': 997, 'humidity': 65, 'temp_kf': -1}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'clouds': {'all': 96}, 'wind': {'speed': 2.14, 'deg': 145, 'gust': 2.5}, 'visibility': 10000, 'pop': 0.77, 'sys': {'pod': 'd'}, 'dt_txt': '2025-08-09 06:00:00'}, {'dt': 1754730000, 'main': {'temp': 27.01, '

In [12]:
forecast_time = "2025-04-05 14:00:00"
forecast_data = None
for forecast in weather_data["list"]:
    if forecast["dt_txt"] == forecast_time:
        forecast_data = forecast
        break

if forecast_data:
    rain_probability = forecast_data["pop"]
    temperature = forecast_data["main"]["temp"]  
else:
    rain_probability = 0 
    temperature = 20 

In [13]:
merged_data = qualifying_2025.merge(sector_times_2024, left_on="Driver", right_on="Driver", how="left")

In [14]:
# Create weather features for the model
merged_data["RainProbability"] = rain_probability
merged_data["Temperature"] = temperature

In [15]:
X = merged_data[["QualifyingTime (s)", "Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)", "WetPerformanceFactor", "RainProbability", "Temperature"]].fillna(0)


In [16]:
y = merged_data.merge(laps_2024.groupby("Driver")["LapTime (s)"].mean(), left_on="Driver", right_index=True)["LapTime (s)"]


In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=38)
model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=38)
model.fit(X_train, y_train)

GradientBoostingRegressor(n_estimators=200, random_state=38)

In [18]:
predicted_race_times = model.predict(X)
qualifying_2025["PredictedRaceTime (s)"] = predicted_race_times
qualifying_2025 = qualifying_2025.sort_values(by="PredictedRaceTime (s)")

In [19]:
print("\n🏁 Predicted 2025 Japanese GP Winner🏁\n")
print(qualifying_2025[["Driver", "PredictedRaceTime (s)"]])


🏁 Predicted 2025 Japanese GP Winner🏁

   Driver  PredictedRaceTime (s)
0     VER              96.975560
3     LEC              97.417480
1     NOR              97.531600
4     RUS              97.766740
5     HAM              97.847000
2     PIA              97.849040
9     SAI              97.873861
7     ALO              97.917323
10    HUL              98.523858
8     TSU              98.865694
12    STR              99.019673
11    OCO             100.458840
6     GAS             100.673660


In [20]:
y_pred = model.predict(X_test)
print(f"\n🔍 Model Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} seconds")


🔍 Model Error (MAE): 0.34 seconds
